# Sample Model -- Used by Streamlit

## Recorded Results

The Random Forest model was reduced from 300 trees to 8 trees to significantly reduce the model size and enable deployment through Streamlit.

| Version | Accuracy | Macro F1 | Weighted F1 | Model Size |
| --- | ---: | ---: | ---: | ---: |
| 300 Trees | 86.15% | 78.74% | 85.37% | 7 GB |
| 100 Trees | 85.60% | 79.66% | 84.75% | 308 MB |
| **8 Trees — Streamlit Model** | **80.31%** | **72.80%** | **79.38%** | **25.15 MB** |



In [3]:
import time
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "coastalcph/lex_glue",
    "ledgar"
)

In [2]:
print(dataset)

for split in dataset:
    print(split, len(dataset[split]))

print(dataset["train"].features)

print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
})
train 60000
test 10000
validation 10000
{'text': Value('string'), 'label': ClassLabel(names=['Adjustments', 'Agreements', 'Amendments', 'Anti-Corruption Laws', 'Applicable Laws', 'Approvals', 'Arbitration', 'Assignments', 'Assigns', 'Authority', 'Authorizations', 'Base Salary', 'Benefits', 'Binding Effects', 'Books', 'Brokers', 'Capitalization', 'Change In Control', 'Closings', 'Compliance With Laws', 'Confidentiality', 'Consent To Jurisdiction', 'Consents', 'Construction', 'Cooperation', 'Costs', 'Counterparts', 'Death', 'Defined Terms', 'Definitions', 'Disability', 'Disclosures', 'Duties', 'Effective Dates', 'Effectiveness', 'Employment', 'Enforceability', 'Enforcements', 'Entire Agreements', 'Erisa',

In [4]:
train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

X_train = train["text"]
y_train = train["label"]

X_validation = validation["text"]
y_validation = validation["label"]

X_test = test["text"]
y_test = test["label"]

print("Training examples:", len(X_train))
print("Validation examples:", len(X_validation))
print("Test examples:", len(X_test))

Training examples: 60000
Validation examples: 10000
Test examples: 10000


In [5]:
# Step 2 — Convert text into TF-IDF
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

start_time = time.time()

X_train_tfidf = tfidf.fit_transform(X_train)

X_validation_tfidf = tfidf.transform(X_validation)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_validation_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

print("TF-IDF vocabulary size:", len(tfidf.vocabulary_))
print("Transformation time:", round(time.time() - start_time, 2), "seconds")

TF-IDF training shape: (60000, 210328)
TF-IDF validation shape: (10000, 210328)
TF-IDF test shape: (10000, 210328)
TF-IDF vocabulary size: 210328
Transformation time: 8.31 seconds


# Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

rf_model.fit(
    X_train_tfidf,
    y_train
)

rf_training_time = time.time() - start_time

print("Random Forest training completed.")
print("Training time:", round(rf_training_time, 2), "seconds")

Random Forest training completed.
Training time: 9.83 seconds


In [7]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Make predictions
y_pred_rf = rf_model.predict(X_test_tfidf)

# Calculate metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_macro_f1 = f1_score(y_test, y_pred_rf, average="macro")
rf_weighted_f1 = f1_score(y_test, y_pred_rf, average="weighted")

print("Random Forest Results")
print("---------------------")
print(f"Accuracy:    {rf_accuracy:.4f}")
print(f"Macro F1:    {rf_macro_f1:.4f}")
print(f"Weighted F1: {rf_weighted_f1:.4f}")

Random Forest Results
---------------------
Accuracy:    0.8031
Macro F1:    0.7280
Weighted F1: 0.7938


In [8]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.83        88
           1       0.47      0.35      0.40        48
           2       0.63      0.87      0.73       224
           3       0.85      1.00      0.92        23
           4       0.40      0.19      0.26        53
           5       0.58      0.58      0.58        26
           6       0.71      0.96      0.82        47
           7       0.67      0.84      0.75       195
           8       1.00      0.25      0.40         4
           9       0.47      0.66      0.55        62
          10       0.71      0.68      0.69        90
          11       0.85      0.97      0.91       112
          12       0.70      0.62      0.66        81
          13       0.65      0.72      0.68       126
          14       1.00      1.00      1.00         2
          15       0.89      1.00      0.94        70
          16       0.89      0.92      0.91        63
   

In [19]:
import os
import joblib

model_path = "legal_clause_random_forest_8.joblib"

joblib.dump(
    rf_model,
    model_path,
    compress=3
)

model_size_mb = os.path.getsize(model_path) / (1024 ** 2)

print(f"Saved model: {model_path}")
print(f"Model size: {model_size_mb:.2f} MB")

Saved model: legal_clause_random_forest_8.joblib
Model size: 24.62 MB


In [10]:
# Delete the current model from memory
del rf_model

# Reload the saved model
rf_model_test = joblib.load(model_path)

print("Model successfully reloaded.")
print("Number of trees:", len(rf_model_test.estimators_))

Model successfully reloaded.
Number of trees: 8


In [11]:
y_pred_saved = rf_model_test.predict(X_test_tfidf)

saved_accuracy = accuracy_score(y_test, y_pred_saved)

print(f"Reloaded model accuracy: {saved_accuracy:.4f}")

Reloaded model accuracy: 0.8031


In [12]:
final_model_path = "legal_clause_random_forest.joblib"

joblib.dump(
    rf_model_test,
    final_model_path,
    compress=3
)

final_size_mb = os.path.getsize(final_model_path) / (1024 ** 2)

print(f"Final model saved: {final_model_path}")
print(f"Final model size: {final_size_mb:.2f} MB")

Final model saved: legal_clause_random_forest.joblib
Final model size: 25.15 MB


In [13]:
import joblib
import os

# Load the exact final model
rf_deploy = joblib.load("legal_clause_random_forest.joblib")

print("Model loaded successfully.")
print("Number of trees:", len(rf_deploy.estimators_))

# Test prediction
y_pred_deploy = rf_deploy.predict(X_test_tfidf)

print("Prediction successful.")
print("Accuracy:", accuracy_score(y_test, y_pred_deploy))
print("Macro F1:", f1_score(y_test, y_pred_deploy, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_deploy, average="weighted"))

Model loaded successfully.
Number of trees: 8
Prediction successful.
Accuracy: 0.8031
Macro F1: 0.7279849924484132
Weighted F1: 0.7938275758772838


# Test the Model

In [ ]:
# ==========================================
# Test 5 Example Legal Clauses
# ==========================================

import pandas as pd
import numpy as np

label_names = [
    "Adjustments", "Agreements", "Amendments", "Anti-Corruption Laws",
    "Applicable Laws", "Approvals", "Arbitration", "Assignments",
    "Assigns", "Authority", "Authorizations", "Base Salary",
    "Benefits", "Binding Effects", "Books", "Brokers", "Capitalization",
    "Change In Control", "Closings", "Compliance With Laws",
    "Confidentiality", "Consent To Jurisdiction", "Consents",
    "Construction", "Cooperation", "Costs", "Counterparts", "Death",
    "Defined Terms", "Definitions", "Disability", "Disclosures", "Duties",
    "Effective Dates", "Effectiveness", "Employment", "Enforceability",
    "Enforcements", "Entire Agreements", "Erisa", "Existence", "Expenses",
    "Fees", "Financial Statements", "Forfeitures", "Further Assurances",
    "General", "Governing Laws", "Headings", "Indemnifications",
    "Indemnity", "Insurances", "Integration", "Intellectual Property",
    "Interests", "Interpretations", "Jurisdictions", "Liens", "Litigations",
    "Miscellaneous", "Modifications", "No Conflicts", "No Defaults",
    "No Waivers", "Non-Disparagement", "Notices", "Organizations",
    "Participations", "Payments", "Positions", "Powers", "Publicity",
    "Qualifications", "Records", "Releases", "Remedies", "Representations",
    "Sales", "Sanctions", "Severability", "Solvency", "Specific Performance",
    "Submission To Jurisdiction", "Subsidiaries", "Successors", "Survival",
    "Tax Withholdings", "Taxes", "Terminations", "Terms", "Titles",
    "Transactions With Affiliates", "Use Of Proceeds", "Vacations", "Venues",
    "Vesting", "Waiver Of Jury Trials", "Waivers", "Warranties",
    "Withholdings"
]

# Five example legal clauses
test_examples = [
    {
        "text": "The employee shall receive a base salary of $100,000 per year, payable in accordance with the Company's regular payroll practices.",
        "expected": "Base Salary"
    },
    {
        "text": "The employee agrees not to disclose any confidential information of the Company to any person or entity except as required by law.",
        "expected": "Confidentiality"
    },
    {
        "text": "Any dispute arising out of or relating to this Agreement shall be resolved by binding arbitration in accordance with the applicable arbitration rules.",
        "expected": "Arbitration"
    },
    {
        "text": "This Agreement shall be governed by and construed in accordance with the laws of the State of New York, without regard to its conflict of laws principles.",
        "expected": "Governing Laws"
    },
    {
        "text": "Each party hereby waives any right to trial by jury in any action or proceeding arising out of or relating to this Agreement.",
        "expected": "Waiver Of Jury Trials"
    }
]

results = []

for i, example in enumerate(test_examples, 1):

    # Convert text to TF-IDF features
    text_tfidf = tfidf.transform([example["text"]])

    # Get class probabilities
    probabilities = rf_model.predict_proba(text_tfidf)[0]

    # Get predicted class
    predicted_index = np.argmax(probabilities)
    predicted_label = label_names[predicted_index]

    # Get confidence
    confidence = probabilities[predicted_index]

    # Check prediction
    correct = predicted_label == example["expected"]

    results.append({
        "Test": i,
        "Expected": example["expected"],
        "Predicted": predicted_label,
        "Confidence": f"{confidence:.2%}",
        "Correct": "✓" if correct else "✗"
    })

# Create results table
results_df = pd.DataFrame(results)

print("Five Example Legal Clause Predictions")
print("=" * 80)
print(results_df.to_string(index=False))

# Calculate test accuracy
test_accuracy = sum(
    result["Correct"] == "✓"
    for result in results
) / len(results)

print("\nFive-Example Accuracy:", f"{test_accuracy:.2%}")

Five Example Legal Clause Predictions
 Test              Expected             Predicted Confidence Correct
    1           Base Salary           Base Salary     87.50%       ✓
    2       Confidentiality       Confidentiality     87.50%       ✓
    3           Arbitration           Arbitration     37.50%       ✓
    4        Governing Laws        Governing Laws     87.50%       ✓
    5 Waiver Of Jury Trials Waiver Of Jury Trials     87.50%       ✓

Five-Example Accuracy: 100.00%


In [18]:
# ==========================================
# Test 5 Additional Legal Clauses
# ==========================================

test_examples = [
    {
        "text": "This contract shall be interpreted under the laws of the Commonwealth of Virginia, without regard to its conflict-of-law principles.",
        "expected": "Governing Laws"
    },
    {
        "text": "Neither party may disclose proprietary information received during the performance of this agreement to any person outside its organization unless disclosure is required by applicable law.",
        "expected": "Confidentiality"
    },
    {
        "text": "If any provision of this agreement is determined to be invalid or unenforceable, the remaining provisions shall continue in full force and effect.",
        "expected": "Severability"
    },
    {
        "text": "All formal notices under this agreement must be delivered electronically to the addresses designated by the parties and shall become effective upon confirmed receipt.",
        "expected": "Notices"
    },
    {
        "text": "A party's failure to exercise a contractual right immediately shall not prevent that party from exercising the same right at a later date.",
        "expected": "No Waivers"
    }
]

results = []

for i, example in enumerate(test_examples, 1):

    # Convert text to TF-IDF features
    text_tfidf = tfidf.transform([example["text"]])

    # Get prediction probabilities
    probabilities = rf_model.predict_proba(text_tfidf)[0]

    # Get predicted class
    predicted_index = np.argmax(probabilities)
    predicted_label = label_names[predicted_index]

    # Get confidence
    confidence = probabilities[predicted_index]

    # Check whether prediction is correct
    correct = predicted_label == example["expected"]

    results.append({
        "Test": i,
        "Expected": example["expected"],
        "Predicted": predicted_label,
        "Confidence": f"{confidence:.2%}",
        "Correct": "✓" if correct else "✗"
    })

results_df = pd.DataFrame(results)

print("Additional Legal Clause Predictions")
print("=" * 85)
print(results_df.to_string(index=False))

# Calculate accuracy
test_accuracy = sum(
    result["Correct"] == "✓"
    for result in results
) / len(results)

print("\nFive-Example Accuracy:", f"{test_accuracy:.2%}")

Additional Legal Clause Predictions
 Test        Expected       Predicted Confidence Correct
    1  Governing Laws  Governing Laws     62.50%       ✓
    2 Confidentiality     Disclosures     37.50%       ✗
    3    Severability    Severability    100.00%       ✓
    4         Notices Effective Dates     62.50%       ✗
    5      No Waivers      No Waivers     25.00%       ✓

Five-Example Accuracy: 60.00%
